# 03b — XGBoost

Dette notebooket trener XGBoost på samme klargjorte ISO-grunnlag som OLS.

**Input:** `intermediate/df_iso.parquet`

**Output:** `intermediate/preds_xgb.parquet`, `intermediate/models_xgb.pkl`


In [1]:
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor

from src.config import (
    INTERMEDIATE_DIR, TARGET, XGB_FEATURES, TRAIN_YEARS, TEST_YEARS, apply_style
)
from src.evaluation import eval_metrics
from src.model_training import (
    load_prepared_iso_data, split_features_target, make_prediction_frame,
    save_model_artifacts,
)

apply_style()


In [2]:
df_iso = load_prepared_iso_data(INTERMEDIATE_DIR)
X_train, y_train, X_test, y_test, train_mask, test_mask = split_features_target(
    df_iso,
    feature_cols=XGB_FEATURES,
    target=TARGET,
    train_years=TRAIN_YEARS,
    test_years=TEST_YEARS,
)

print(f"Trening: {len(X_train):,} ISO-timer")
print(f"Test:    {len(X_test):,} ISO-timer")
print(f"Features: {len(XGB_FEATURES)}")


Trening: 25,072 ISO-timer
Test:    9,844 ISO-timer
Features: 37


In [3]:
ts_cv = TimeSeriesSplit(n_splits=4)
param_grid = [
    {"max_depth": 3, "n_estimators": 200, "learning_rate": 0.05},
    {"max_depth": 4, "n_estimators": 300, "learning_rate": 0.05},
    {"max_depth": 4, "n_estimators": 500, "learning_rate": 0.03},
]

cv_rows = []
for params in param_grid:
    fold_maes = []
    for fold, (tr_idx, val_idx) in enumerate(ts_cv.split(X_train), start=1):
        model = XGBRegressor(
            **params,
            objective="reg:squarederror",
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            verbosity=0,
        )
        model.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
        val_pred = model.predict(X_train.iloc[val_idx])
        mae = (y_train.iloc[val_idx] - val_pred).abs().mean()
        fold_maes.append(mae)
        cv_rows.append({"fold": fold, **params, "MAE": mae})

cv_df = pd.DataFrame(cv_rows)
cv_summary = (
    cv_df.groupby(["max_depth", "n_estimators", "learning_rate"], as_index=False)["MAE"]
    .mean()
    .sort_values("MAE")
)
best_params = cv_summary.iloc[0][["max_depth", "n_estimators", "learning_rate"]].to_dict()
best_params["max_depth"] = int(best_params["max_depth"])
best_params["n_estimators"] = int(best_params["n_estimators"])

display(cv_summary)
print(f"Beste parametre: {best_params}")


,max_depth,n_estimators,learning_rate,MAE
0,3,200,0.05,216.154874
1,4,300,0.05,227.072848
2,4,500,0.03,231.459150


Beste parametre: {'max_depth': 3, 'n_estimators': 200, 'learning_rate': 0.05}


In [4]:
xgb_model = XGBRegressor(
    **best_params,
    objective="reg:squarederror",
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0,
)
xgb_model.fit(X_train, y_train)

preds = make_prediction_frame(
    df=df_iso,
    mask=test_mask,
    actual=y_test,
    prediction_col="XGB",
    predictions=xgb_model.predict(X_test),
)
metrics = eval_metrics(preds["actual"], preds["XGB"])
display(pd.DataFrame([metrics], index=["XGB"]))


,MAE,RMSE,R²,N
XGB,180.7,264.5,-0.0679,9844


In [5]:
payload = {
    "model_name": "XGB",
    "prediction_col": "XGB",
    "feature_cols": XGB_FEATURES,
    "target_col": TARGET,
    "train_years": TRAIN_YEARS,
    "test_years": TEST_YEARS,
    "estimator": xgb_model,
    "metrics": metrics,
    "best_params": best_params,
    "cv_summary": cv_summary,
    "cv_rows": cv_df,
}

save_model_artifacts("xgb", preds, payload, intermediate_dir=INTERMEDIATE_DIR)
print(f"Lagret {INTERMEDIATE_DIR}preds_xgb.parquet")
print(f"Lagret {INTERMEDIATE_DIR}models_xgb.pkl")


Lagret intermediate/preds_xgb.parquet
Lagret intermediate/models_xgb.pkl
